## UPSC Essay evaluation Workflow

In [32]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel,Field
from typing import List, Dict, Annotated
from dotenv import load_dotenv
import operator
import os
load_dotenv()

True

### Structured Output defining

In [33]:
#  here we are going to use pydantic for structured output from LLM models 

# defining the schema for evaluation
class EvaluationSchema(BaseModel):
    score: int = Field( description="Score between 1 and 10",ge=0, le=10)
    feedback: str = Field( description="Detailed feedback on the essay")

model = ChatGoogleGenerativeAI(api_key=os.getenv("GOOGLE_API_KEY"), model="gemini-2.5-flash", temperature=0.2)

structured_evaluation_model = model.with_structured_output(EvaluationSchema)

### State defining for this Problem 

In [34]:
class EvaluationState(BaseModel):
    essay: str=""
    cot_feedback:str=""
    doa_feedback:str=""
    lang_feedback:str=""

    final_feedback: str=""
    individual_scores: Annotated[List[int],operator.add]=[]
    average_score: float=0.0


# shit ye Annotated- used to attach extra meta data with changing the type and List - check each element in List is int type 
# and operator.add - reducer function 
# basically each COT,DOA,Lang is executed paralley and score generated by them is parallely generated and
# chance of overwriting more 
# so to reduce this overwriting we use reducer function (operator.add) 
# we can't use + here 
# each have there own score [8],[7],[6]
# reducer we just [8],[7],[6] -> [8.7.6] final score list 


### Functionality of each node

In [35]:
# functions of each Nodes

# calrity of thought function
def evaluate_cot(state: EvaluationState):
    # prompt 
    prompt = f"Evaluate the clarity of thought of following essay and provide the detailed feedback and score for this essay: {state.essay}"
    evaluation_output = structured_evaluation_model.invoke(prompt)
    return {'cot_feedback': evaluation_output.feedback,'individual_scores':[ evaluation_output.score]}

# depth of analysis function
def evaluate_doa(state: EvaluationState):
    # prompt
    prompt = f"Evaluate the depth of analysis of following essay and provide the detailed feedback and score for this essay: {state.essay}"
    evaluation_output = structured_evaluation_model.invoke(prompt)
    return {'doa_feedback': evaluation_output.feedback,'individual_scores':[ evaluation_output.score]}


# language quality function
def evaluate_lang(state: EvaluationState):
    # prompt
    prompt = f"Evaluate the language quality of following essay and provide the detailed feedback and score for this essay: {state.essay}"
    evaluation_output = structured_evaluation_model.invoke(prompt)
    return {'lang_feedback': evaluation_output.feedback,'individual_scores':[ evaluation_output.score]}


# final feedback function
def generate_final_feedback(state: EvaluationState):

    # final_feedback prompt
    prompt = f"""
    Provide a comprehensive feedback for the essay based on the following individual feedbacks:
    Clarity of Thought Feedback: {state.cot_feedback}
    Depth of Analysis Feedback: {state.doa_feedback}
    Language Quality Feedback: {state.lang_feedback}
    """
    
    final_feedback = model.invoke(prompt).content

    # average score calculation
    average_score = sum(state.individual_scores)/len(state.individual_scores)

    
    return {'final_feedback': final_feedback,'average_score': average_score}

### Graph structuring 

In [36]:
graph = StateGraph(EvaluationState)


# adding nodes to graph 
graph.add_node('evaluate_cot',evaluate_cot)
graph.add_node('evaluate_doa',evaluate_doa)
graph.add_node('evaluate_lang',evaluate_lang)
graph.add_node('generate_final_feedback',generate_final_feedback)

# defining edges
graph.add_edge(START,'evaluate_cot')
graph.add_edge(START,'evaluate_doa')
graph.add_edge(START,'evaluate_lang')

graph.add_edge('evaluate_cot','generate_final_feedback')
graph.add_edge('evaluate_doa','generate_final_feedback')
graph.add_edge('evaluate_lang','generate_final_feedback') 

graph.add_edge('generate_final_feedback',END)

workflow = graph.compile()

In [37]:
essay_text_1= """
Artificial Intelligence (AI) has emerged as a transformative force in modern education, reshaping how knowledge is delivered, assessed, and personalized. As digital technologies continue to evolve, AI has moved beyond being a supplementary tool and has become an integral component of educational systems worldwide. Its impact can be seen in personalized learning, administrative efficiency, and the development of future-ready skills.

One of the most significant contributions of AI to education is personalized learning. Traditional classroom models often follow a one-size-fits-all approach, which fails to address the diverse learning needs of students. AI-powered systems analyze student performance, learning speed, and preferences to create customized learning paths. Adaptive learning platforms adjust content difficulty in real time, ensuring that advanced learners remain challenged while struggling students receive additional support. This individualized approach not only improves academic outcomes but also increases student engagement and motivation.

In addition to personalized learning, AI has greatly enhanced assessment and feedback mechanisms. Automated grading systems can evaluate objective tests instantly, saving valuable time for educators. More advanced AI tools can analyze written responses, identify patterns in mistakes, and provide constructive feedback. This allows teachers to focus on higher-order tasks such as mentoring, curriculum design, and emotional support, rather than repetitive administrative work.

AI also plays a crucial role in improving accessibility and inclusivity in education. Speech-to-text tools, language translation systems, and AI-powered tutors help students with disabilities or language barriers participate more effectively in learning environments. For students in remote or under-resourced regions, AI-based online education platforms offer access to high-quality learning materials that were previously unavailable.

However, the integration of AI into education is not without challenges. Concerns related to data privacy, algorithmic bias, and over-reliance on technology must be addressed carefully. Educational institutions must ensure that student data is protected and that AI systems are transparent and fair. Moreover, AI should complement human educators rather than replace them, as emotional intelligence, ethical judgment, and creativity remain uniquely human qualities.

In conclusion, Artificial Intelligence has the potential to revolutionize modern education by making learning more personalized, efficient, and inclusive. While challenges exist, thoughtful implementation and ethical governance can ensure that AI serves as a powerful ally in shaping the future of education. When used responsibly, AI can empower both students and educators, leading to a more adaptive and effective educational system.

"""



essay_text_2 = """
Artificial Intelligence is becoming very popular in many fields, and education is one of them. AI is being used in schools and colleges to help students learn better. Many people believe that AI is useful, but some people think it can create problems. Overall, AI has changed education in different ways.

One important use of AI in education is online learning. Many students now study using online platforms that use artificial intelligence. These platforms provide videos, notes, and tests. AI helps students by giving them questions and checking answers. This makes learning easier and faster. Students can learn from their homes without going to school. This is very helpful, especially during situations like pandemics.

AI is also used for checking exams and assignments. Teachers do not have to check every paper manually. AI systems can quickly check answers and give marks. This saves time and effort. Teachers can use this time for teaching students better. However, sometimes AI may not understand answers properly, which can be a problem.

Another advantage of AI is that it helps weak students. AI systems can find which students are weak and give them extra practice. This can improve their performance. At the same time, bright students can get harder questions. This is helpful, but it depends a lot on technology.

There are also some disadvantages of AI in education. One major issue is data privacy. Student information is stored online, and it can be misused. Also, students may depend too much on AI and stop thinking on their own. Teachers may also lose their importance if AI is used too much.

In conclusion, Artificial Intelligence has both advantages and disadvantages in education. It helps students learn better and saves teachers’ time, but it also creates problems like data security and over-dependence. AI should be used carefully so that education does not lose its human touch.
"""

In [ ]:
initial_state = {
    "essay":essay_text_2
}

final_state = workflow.invoke(initial_state)

In [40]:
final_state['final_feedback'], final_state['average_score']

('This essay demonstrates a remarkable foundation, showcasing exceptional clarity of thought and excellent language quality. It provides a well-structured and balanced overview of Artificial Intelligence\'s impact on modern education, effectively covering both its benefits and associated challenges.\n\nHere\'s a comprehensive breakdown of the feedback:\n\n### Overall Strengths\n\nThe essay\'s most significant strengths lie in its **clarity of thought** and **language quality**. The arguments are presented with outstanding structure and logical progression, making the essay exceptionally easy to follow. Each paragraph is meticulously crafted with clear topic sentences, thoroughly supported by relevant explanations and examples. The consistent focus on AI\'s impact in education, coupled with seamless transitions and a balanced perspective, demonstrates a comprehensive understanding of the subject. There is no ambiguity, abruptness, or convoluted reasoning, highlighting a strong command o